# Load dataset

Press `Run All` from this notebook for loading Milvus Database

In [3]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("gvaldenebro/cancer-q-and-a-dataset")

csv_files = []

for dirs, _, files in os.walk(path):
    for file in files:
      csv_files.append(os.path.join(dirs, file))

csv_files

/Users/trantrunghcmut/Documents/hcmut/internship/Chatbot Y Khoa/chatbot/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['/Users/trantrunghcmut/.cache/kagglehub/datasets/gvaldenebro/cancer-q-and-a-dataset/versions/4/growth_hormone_receptorQA.csv',
 '/Users/trantrunghcmut/.cache/kagglehub/datasets/gvaldenebro/cancer-q-and-a-dataset/versions/4/SeniorHealthQA.csv',
 '/Users/trantrunghcmut/.cache/kagglehub/datasets/gvaldenebro/cancer-q-and-a-dataset/versions/4/Heart_Lung_and_BloodQA.csv',
 '/Users/trantrunghcmut/.cache/kagglehub/datasets/gvaldenebro/cancer-q-and-a-dataset/versions/4/Diabetes_and_Digestive_and_Kidney_DiseasesQA.csv',
 '/Users/trantrunghcmut/.cache/kagglehub/datasets/gvaldenebro/cancer-q-and-a-dataset/versions/4/Neurological_Disorders_and_StrokeQA.csv',
 '/Users/trantrunghcmut/.cache/kagglehub/datasets/gvaldenebro/cancer-q-and-a-dataset/versions/4/Genetic_and_Rare_DiseasesQA.csv',
 '/Users/trantrunghcmut/.cache/kagglehub/datasets/gvaldenebro/cancer-q-and-a-dataset/versions/4/CancerQA.csv',
 '/Users/trantrunghcmut/.cache/kagglehub/datasets/gvaldenebro/cancer-q-and-a-dataset/versions/4/Disease_

## Global Variable

In [ ]:
import dotenv
dotenv.load_dotenv()

MAX_TOKEN_BATCH = 175000        # Maxium number of tokens per batch
API_KEY = dotenv.get_key("OPENAI_API_KEY")
EMBEDDINGS_MODEL = "text-embedding-3-small"  # OpenAI Embeddings model

MILVUS_BATCH_SIZE = 1000

## Read data

In [106]:
import pandas as pd
from tqdm import tqdm

use_cols = ['Question', 'Answer', 'topic']

full_df = pd.DataFrame(columns=use_cols)

for file in tqdm(csv_files):
  sample_df = pd.read_csv(file)
  full_df = pd.concat([full_df, sample_df], axis=0)


full_df['count_word'] = full_df.apply(lambda x: len(x['Question'].split()) + len(x['Answer'].split()), axis=1)
full_df['Text'] = full_df.apply(lambda x: '**Question**: ' + x['Question'] + '\n\n **Answer:** ' + x['Answer'], axis=1)

print("Total tokens: ", full_df['count_word'].sum())

full_df.drop_duplicates(subset=['Text'], inplace=True)

full_df.info()

100%|██████████| 10/10 [00:00<00:00, 42.67it/s]


Total tokens:  6876200
<class 'pandas.core.frame.DataFrame'>
Index: 16358 entries, 0 to 980
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Question    16358 non-null  object
 1   Answer      16358 non-null  object
 2   topic       16358 non-null  object
 3   split       16358 non-null  object
 4   count_word  16358 non-null  int64 
 5   Text        16358 non-null  object
dtypes: int64(1), object(5)
memory usage: 894.6+ KB


## Split data into chunks for embedding by batch

In [108]:
from typing import List

class TextLoader:
    def __init__(self, texts: List[str], token_count: List[str]):
        self.texts = texts
        self.token_count = token_count

    def __iter__(self):
        current_batch = []
        current_token_count = 0

        for text, tokens in zip(self.texts, self.token_count):
            if current_token_count + tokens > MAX_TOKEN_BATCH:
                yield current_batch
                current_batch = []
                current_token_count = 0
            
            current_batch.append(text)
            current_token_count += tokens
        
        if current_batch:
            yield current_batch

text_loaders = TextLoader(full_df['Text'].tolist(), full_df['count_word'].tolist())
batchs = [batch for batch in text_loaders]

len(batchs), batchs[0][:5]  # Show number of batches and first 5 items in the first batch

(20,
 ['**Question**: What is (are) keratoderma with woolly hair ?\n\n **Answer:** Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the soles of the feet to become thick, scaly, and calloused.  Cardiomyopathy, which is a disease of the heart muscle, is a life-threatening health problem that can develop in people with keratoderma with woolly hair. Unlike the other features of this condition, signs and symptoms of cardiomyopathy may not appear until adolescence or later. Complications of cardiomyopathy can

# 2. Prepare models

## Prepare OpenAI Model

In [109]:
# from openai import OpenAI, RateLimitError
from langchain_openai import OpenAIEmbeddings
from langchain_core.embeddings import Embeddings
from langchain_openai.embeddings import OpenAIEmbeddings
from time import sleep

model = OpenAIEmbeddings(model="text-embedding-3-small", api_key=API_KEY)

embeddings = []
sleep_count = 5

# Loader: batch - list of texts used for embedding
for batch in tqdm(batchs):
    if sleep_count <= 0:
        print("Sleeping for 60 seconds to avoid rate limit...")
        sleep(70)
        sleep_count = 5         # Reset sleep count after sleep
    # Embedding
    batch_embeddings = model.embed_documents(batch)
    embeddings.extend(batch_embeddings)
    sleep_count -= 1

 25%|██▌       | 5/20 [00:54<02:30, 10.03s/it]

Sleeping for 60 seconds to avoid rate limit...


 50%|█████     | 10/20 [02:45<02:20, 14.06s/it]

Sleeping for 60 seconds to avoid rate limit...


 75%|███████▌  | 15/20 [04:36<01:12, 14.44s/it]

Sleeping for 60 seconds to avoid rate limit...


100%|██████████| 20/20 [06:17<00:00, 18.85s/it]


In [110]:
saved_embeddings = embeddings.copy()  # Save embeddings to a variable for later use

len(embeddings)

16358

# 3. Load into Milvus Database

In [1]:
from pymilvus import connections, utility, Collection, FieldSchema, DataType, CollectionSchema

con = connections.connect(
    alias="default",
    host="localhost",
    port="19530"
)

utility.list_collections()

['medical_QA_embedding', 'recursive_TE3_embedding']

In [ ]:
max(len(answer) for answer in full_df['Answer'].tolist()), max(len(question) for question in full_df['Question'].tolist())

## Create Schema

In [122]:
FIELDS = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="embedding_vector", dtype=DataType.FLOAT_VECTOR, dim=1536),
    FieldSchema(name="question", dtype=DataType.VARCHAR, max_length=200),
    FieldSchema(name="answer", dtype=DataType.VARCHAR, max_length=30000),
    FieldSchema(name="topic", dtype=DataType.VARCHAR, max_length=100),
]

RAG_SCHEMA = CollectionSchema(fields=FIELDS, description="Medical Chatbot Vector Database")

collection = Collection(name="medical_QA_embedding", schema=RAG_SCHEMA)
collection.create_index(
    field_name="embedding_vector", 
    index_type="IVF_FLAT", 
    metric_type="COSINE",
    index_name="vector_index",
)

data = [
    {
        "embedding_vector": embedding,
        "question": question,
        "answer": answer,  # Assuming 'text' is the combined question and answer
        "topic": topic,
    }
    for embedding, question, answer, topic in zip(embeddings, full_df['Question'].to_list(), full_df['Answer'].to_list(), full_df['topic'].to_list())
]

for i in tqdm(range(0, len(data), MILVUS_BATCH_SIZE)):
    batch = data[i:i+MILVUS_BATCH_SIZE]
    collection.insert(batch)

print("All batches inserted.")

collection.load()

utility.list_collections()

100%|██████████| 17/17 [00:14<00:00,  1.16it/s]


All batches inserted.


['medical_QA_embedding', 'recursive_TE3_embedding']

## Test Query for dataset

In [123]:
def query(query: str, n_results: int = 3):
    query_embedding = model.embed_documents([query])

    data = collection.search(
        data=query_embedding,
        anns_field="embedding_vector",
        limit=n_results,
        param={"metric_type": "COSINE"},
        output_fields=["answer", "topic"],
        consistency_level="Strong"
    )

    return data

In [126]:
query_results = query(query="What is keratoderma with woolly hair ?", n_results=10)

query_results = [(res.get('answer'), res.get('topic')) for res in query_results[0]]

query_results

[('Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the soles of the feet to become thick, scaly, and calloused.  Cardiomyopathy, which is a disease of the heart muscle, is a life-threatening health problem that can develop in people with keratoderma with woolly hair. Unlike the other features of this condition, signs and symptoms of cardiomyopathy may not appear until adolescence or later. Complications of cardiomyopathy can include an abnormal heartbeat (arrhythmia), heart failure, and sudden death.  K